In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler


In [2]:
import joblib

model = joblib.load("C:\\Users\\krish\\OneDrive\\Desktop\\RedditTrendPredictor\\models\\trend_model.pkl")
vectorizer = joblib.load("C:\\Users\\krish\\OneDrive\\Desktop\\RedditTrendPredictor\\models\\tfidf_vectorizer.pkl")
scaler = joblib.load("C:\\Users\\krish\\OneDrive\\Desktop\\RedditTrendPredictor\\models\\scaler.pkl")

In [3]:
import praw
import pandas as pd
from praw.models import MoreComments
from textblob import TextBlob

reddit = praw.Reddit(
    client_id="eLYwGkJLwU-_N9DeCK5c1A",
    client_secret="Lfkyhrp6rWjC0gFcx-AnOPlu28-HRA",
    user_agent="kaggle_reddit_trend_app_v1"
)


In [4]:
def fetch_reddit_data(subreddit_name="technology", limit=200):
    subreddit = reddit.subreddit(subreddit_name)
    posts, comments = [], []

    for post in subreddit.hot(limit=limit):
        posts.append({
            "post_id": post.id,
            "title": post.title,
            "selftext": post.selftext,
            "created_utc": post.created_utc
        })

        post.comments.replace_more(limit=0)
        for c in post.comments.list():
            if isinstance(c, MoreComments):
                continue
            comments.append({
                "post_id": post.id,
                "body": c.body,
                "score": c.score
            })

    return pd.DataFrame(posts), pd.DataFrame(comments)


In [5]:
print(list(scaler.feature_names_in_))


['title_len', 'selftext_len', 'hour', 'dayofweek', 'month', 'sentiment', 'num_comments', 'avg_comment_sentiment', 'avg_comment_score']


In [6]:
def process_comments(df_comments):
    df_comments["sentiment"] = df_comments["body"].apply(
        lambda x: TextBlob(str(x)).sentiment.polarity
    )

    return df_comments.groupby("post_id").agg(
        avg_comment_score=("score", "mean"),
        avg_comment_sentiment=("sentiment", "mean"),
        num_comments=("body", "count")
    ).reset_index()


In [7]:
NUMERIC_FEATURES = list(scaler.feature_names_in_)



def build_features(posts, comment_agg):
    df = posts.merge(comment_agg, on="post_id", how="left")
    df.fillna(0, inplace=True)

    df["created_utc"] = pd.to_datetime(df["created_utc"], unit="s")
    df["hour"] = df["created_utc"].dt.hour
    df["dayofweek"] = df["created_utc"].dt.dayofweek
    df["month"] = df["created_utc"].dt.month

    df["title_len"] = df["title"].astype(str).apply(len)
    df["selftext_len"] = df["selftext"].astype(str).apply(len)
    df["sentiment"] = df["title"].apply(
        lambda x: TextBlob(str(x)).sentiment.polarity
    )

    df["text"] = (df["title"] + " " + df["selftext"]).str.lower()
    return df


In [8]:
print("model:", type(model))
print("vectorizer:", type(vectorizer))
print("scaler:", type(scaler))


model: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
vectorizer: <class 'sklearn.feature_extraction.text.TfidfVectorizer'>
scaler: <class 'sklearn.preprocessing._data.StandardScaler'>


In [9]:
from scipy.sparse import hstack

def predict_trending(df):
    X_text = vectorizer.transform(df["text"])
    X_num = scaler.transform(df[NUMERIC_FEATURES])

    X = hstack([X_num, X_text])
    df["trend_probability"] = model.predict_proba(X)[:, 1]

    return df.sort_values("trend_probability", ascending=False)


In [10]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer

assert isinstance(model, RandomForestClassifier)
assert isinstance(vectorizer, TfidfVectorizer)
assert isinstance(scaler, StandardScaler)


In [11]:
if __name__ == "__main__":
    posts, comments = fetch_reddit_data("technology", limit=200)
    comment_agg = process_comments(comments)
    features = build_features(posts, comment_agg)
    results = predict_trending(features)

    results["trend_probability"] = results["trend_probability"] * 100
    print(results[["title", "trend_probability"]].head(10))


    output_file = "trend_predictions.txt"
    with open(output_file, "w", encoding="utf-8") as f:
            for _, row in results.head(10).iterrows():
                f.write(f"Title: {row['title']}\n")
                f.write(f"Trend Probability: {row['trend_probability']}%\n")
                f.write("-" * 60 + "\n")

    print(f"\nResults saved to {output_file}")

                                                 title  trend_probability
4    Gamers Are Extremely Mad About AI: In-game slo...               64.5
105  Chrome, Edge privacy extensions quietly snarf ...               61.0
129  Racks of AI chips are too damn heavy | Old dat...               60.5
58   Taylor Swift and Sabrina Carpenter AI imperson...               59.0
145  McKinsey to make thousands of layoffs as AI ad...               59.0
104  This is Europe’s secret weapon against Trump: ...               58.0
69   Samsung unveils SOCAMM2 memory, teams up with ...               58.0
43   Nadella's message to Microsoft execs: Get on b...               57.5
79   Trump's rush to build nuclear reactors across ...               56.5
125  Trump's rush to build nuclear reactors across ...               56.5

Results saved to trend_predictions.txt
